Solution Scope

Using an initial "flights" table as follow:

flightPlan table:

Departure       Arrival          FlightNumber   Cost      FlightTime

London          Frankfurt        LH20903        759.44    2 
London          San Francisco    EA87334        159.3     10 
London          New York         LH19681        46.21     8 
London          Paris            LH19618        59.21     1.5 
Frankfurt       Vienna           AU9134         569.92    3 
Frankfurt       New York         LH12375        546.1     9 
Frankfurt       Paris            EH54200        848.58    2 
San Francisco   New York         LH71803        379.27    4 
San Francisco   Vienna           EA10922        105.6     11 
San Francisco   Frankfurt        EH29963        29.48     10 
New York        Paris            AU45243        853.72    8 
New York        Vienna           EA8302         178.95    7 
New York        Franfurt         AU36738        799.23    9.5 
Paris           San Francisco    AU53720        941.36    8.5 
Paris           Vienna           LH89281        873.52    3 
Paris           Frankfurt        EH52253        459.41    2 
Vienna          New York         AU84861        482.42    2.4 
Vienna          Paris            EA37910        74.88     3 
Vienna          Chicago          EH55853        391.23    8

Build a SQL code to reflect a final table within all possible routes within no more than 5 stops departing from Vienna and arriving to all possible destination within stops and its cost.

In [345]:
# Creating table using SQL commands
cursor.execute('''
CREATE TABLE IF NOT EXISTS flights(
    Departure VARCHAR(50),
    Arrival VARCHAR(50),
    FlightNumber VARCHAR(50),
    Cost Float,
    FlightTime Float
    )
''')

# Defining the values to insert
values_to_insert = [
('London', 'Frankfurt', 'LH20903', 759.44, 2),
('London', 'San Francisco', 'EA87334', 159.3, 10),
('London', 'New York', 'LH19681', 46.21, 8),
('London', 'Paris', 'LH19618', 59.21, 1.5),
('Frankfurt', 'Vienna', 'AU9134', 569.92, 3),
('Frankfurt', 'New York', 'LH12375', 546.1, 9),
('Frankfurt', 'Paris', 'EH54200', 848.58, 2),
('San Francisco', 'New York', 'LH71803', 379.27, 4),
('San Francisco', 'Vienna', 'EA10922', 105.6, 11),
('San Francisco', 'Frankfurt', 'EH29963', 29.48, 10),
('New York', 'Paris', 'AU45243', 853.72, 8),
('New York', 'Vienna', 'EA8302', 178.95, 7),
('New York', 'Frankfurt', 'AU36738', 799.23, 9.5),
('Paris', 'San Francisco', 'AU53720', 941.36, 8.5),
('Paris', 'Vienna', 'LH89281', 873.52, 3),
('Paris', 'Frankfurt', 'EH52253', 459.41, 2),
('Vienna', 'New York', 'AU84861', 482.42, 2.4),
('Vienna', 'Paris', 'EA37910', 74.88, 3),
('Vienna', 'Chicago', 'EH55853', 391.23, 8)
]


In [349]:
# Loading data comming from cursor.execute("SELECT * FROM flights")
data = cursor.fetchall()

# Extracting the column names from cursor.description
column_names = [description[0] for description in cursor.description]

# Creating a pandas DataFrame from the fetched data and column names, then display it in a well-formatted table.
flights_df = pd.DataFrame(data, columns=column_names)
flights_df

,Departure,Arrival,FlightNumber,Cost,FlightTime
0,London,Frankfurt,LH20903,759.44,2.0
1,London,San Francisco,EA87334,159.30,10.0
2,London,New York,LH19681,46.21,8.0
3,London,Paris,LH19618,59.21,1.5
4,Frankfurt,Vienna,AU9134,569.92,3.0
5,Frankfurt,New York,LH12375,546.10,9.0
6,Frankfurt,Paris,EH54200,848.58,2.0
7,San Francisco,New York,LH71803,379.27,4.0
8,San Francisco,Vienna,EA10922,105.60,11.0
9,San Francisco,Frankfurt,EH29963,29.48,10.0


In [351]:
# Solution query
sql_cte_query = """
    
CREATE TABLE flight_plan AS

WITH RECURSIVE flight_plan (Departure, Arrival, stops, totalCost, route) AS(
    -- Anchor member: Starts the recursion from 'Vienna'
    SELECT 
        f.Departure, 
        f.Arrival, 
        0 AS stops,
        f.Cost AS totalCost,
        -- Correct fix for concatenation: Explicitly cast as TEXT
        CAST(f.Departure || ' -> ' || f.Arrival AS TEXT) AS route
    FROM flights f
    WHERE f.Departure = 'Vienna'

    UNION ALL

    -- Recursive member: Continues the path to a new, unvisited city
    SELECT 
        p.Departure, 
        f.Arrival, 
        p.stops + 1 AS stops,
        p.totalCost + f.cost AS totalCost,
        p.route || ' -> ' || f.Arrival AS route
    FROM flights f, flight_plan p
    WHERE 
        p.Arrival = f.Departure 
        AND p.stops < 5 
        -- This is the crucial line that prevents cycles:
        -- Ensure the destination is NOT already in the path.
        AND INSTR(p.route, f.Arrival) = 0
)

-- Final SELECT: Displays unique, non-cyclic paths
SELECT
    Departure, 
    Arrival, 
    route,
    stops,
    totalCost
FROM flight_plan
ORDER BY totalCost;                                          
"""

In [357]:
# Loading data comming from cursor.execute('''WITH flight_route (Departure... )
data2 = cursor.fetchall()

# Extracting the column names from cursor.description
column_names2 = [description[0] for description in cursor.description]

# Creating a pandas DataFrame from the fetched data and column names, then display it in a well-formatted table.
flights_route_df = pd.DataFrame(data2, columns=column_names2)
with pd.option_context('display.max_colwidth', None):
    styled_df = flights_route_df.style.format({'totalCost': "{:,.0f}"})
    display(styled_df)

,Departure,Arrival,route,stops,totalCost
0,Vienna,Paris,Vienna -> Paris,0,75
1,Vienna,Chicago,Vienna -> Chicago,0,391
2,Vienna,New York,Vienna -> New York,0,482
3,Vienna,Frankfurt,Vienna -> Paris -> Frankfurt,1,534
4,Vienna,San Francisco,Vienna -> Paris -> San Francisco,1,"1,016"
5,Vienna,Frankfurt,Vienna -> Paris -> San Francisco -> Frankfurt,2,"1,046"
6,Vienna,New York,Vienna -> Paris -> Frankfurt -> New York,2,"1,080"
7,Vienna,Frankfurt,Vienna -> New York -> Frankfurt,1,"1,282"
8,Vienna,Paris,Vienna -> New York -> Paris,1,"1,336"
9,Vienna,New York,Vienna -> Paris -> San Francisco -> New York,2,"1,396"
